# Generative AI for Data Science

## Scenario 1: Retail Inventory Analysis
### Prompt
"I have a retail dataset in a Pandas DataFrame with columns: product_id, category, stock_level, last_restock_date, sales_last_30_days, supplier_lead_time, and unit_cost. Please write a Python script to calculate the inventory turnover rate, identify slow-moving items, and predict potential stockouts."

In [ ]:
import pandas as pd

# Assuming df is already loaded
df['turnover_rate'] = df['sales_last_30_days'] / df['stock_level']
slow_moving = df[df['turnover_rate'] < 0.5]
stockouts = df[df['stock_level'] < 10]

print(slow_moving)
print(stockouts)

### Follow-up Prompts
"The stockout prediction is too basic. Please make it specific by using the supplier_lead_time to see if we will run out before the next shipment arrives. Also, the code crashes if stock_level is 0; please implement error handling for division-by-zero. Finally, add Seaborn visualizations showing stock levels by category and the relationship between turnover and stock."

### Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Sample data for demonstration
np.random.seed(42)
data = {
    'product_id': range(1, 101),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty'], 100),
    'stock_level': np.random.randint(0, 500, 100),
    'sales_last_30_days': np.random.randint(0, 1000, 100),
    'supplier_lead_time': np.random.randint(3, 21, 100),
}
df = pd.DataFrame(data)

# 1. Turnover Rate with zero-division handling
df['turnover_rate'] = df['sales_last_30_days'] / (df['stock_level'] + 1)

# 2. Slow-moving items (bottom 20%)
threshold = df['turnover_rate'].quantile(0.2)
slow_moving = df[df['turnover_rate'] <= threshold]

# 3. Prediction based on lead time
df['daily_sales'] = df['sales_last_30_days'] / 30
df['demand_during_lead'] = df['daily_sales'] * df['supplier_lead_time']
df['stockout_risk'] = df['demand_during_lead'] > df['stock_level']

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(x='category', y='stock_level', data=df, ax=axes[0], estimator=np.mean)
axes[0].set_title('Avg Stock Level by Category')

sns.scatterplot(x='stock_level', y='turnover_rate', hue='category', data=df, ax=axes[1])
axes[1].set_title('Stock Level vs Turnover Rate')

plt.tight_layout()
plt.show()

### Discussion Answers
Date calculations were initially ignored by the AI, but once I pushed for "prediction specificity," it shifted to using a daily sales rate multiplied by the supplier_lead_time to project future needs. For visuals, I ended up with a bar plot for category distribution and a scatter plot to find the "sweet spot" of inventory. The error handling was a simple but necessary addition of + 1 to the denominator to prevent the script from crashing on zero-stock items. Honestly, building out this lead-time logic feels eerily similar to writing Python sales analysis scripts for tracking maize flour business inventory—it's the same basic anxiety of trying to predict exactly when the shelf goes bare.

# Scenario 2: Website Analytics Debug
### Your Prompt
"The marketing team reports that the user engagement metrics code is showing impossible results (bounce rates over 100%, negative session times). Debug this code:

```python
def analyze_user_engagement(logs_df):
    metrics = {
        'bounce_rate': logs_df.groupby('session_id')['page_views'].apply(
            lambda x: x == 1).mean(),
        'avg_session_time': logs_df.groupby('session_id')['duration'].sum(),
        'pages_per_session': logs_df.groupby('session_id')['page_views'].mean()
    }
    
    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })
    
    return metrics, device_metrics
```
"

### Generated Code

In [ ]:
import pandas as pd

def analyze_user_engagement(logs_df):
    # Fix: Group by session first to get total views per session
    session_stats = logs_df.groupby('session_id').agg({
        'page_views': 'sum',
        'duration': 'sum'
    })

    metrics = {
        'bounce_rate': (session_stats['page_views'] == 1).mean(),
        'avg_session_time': session_stats['duration'].mean(),
        'pages_per_session': session_stats['page_views'].mean()
    }

    device_metrics = logs_df.groupby('device_type').agg({
        'session_id': 'count',
        'duration': 'mean',
        'page_views': 'sum'
    })

    return metrics, device_metrics

### Follow-up Prompts
"The bounce rate is fixed, but the 'impossible' negative session times are still there because the raw data has dirty entries. I need explicit data validation at the start of the function to handle NaNs and negative durations. Also, use .nunique() for the session count in device metrics to avoid overcounting rows, and add a Seaborn bar plot to visualize average session time per device."

### Final Solution

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_user_engagement_robust(logs_df):
    # Data Validation
    df = logs_df.copy()
    df['duration'] = df['duration'].fillna(0).clip(lower=0)
    df['page_views'] = df['page_views'].fillna(0)

    # Correct Session Metrics
    session_stats = df.groupby('session_id').agg({
        'page_views': 'sum',
        'duration': 'sum'
    })

    metrics = {
        'bounce_rate': (session_stats['page_views'] == 1).mean(),
        'avg_session_time': session_stats['duration'].mean(),
        'pages_per_session': session_stats['page_views'].mean()
    }

    # Device Metrics with nunique
    device_metrics = df.groupby('device_type').agg({
        'session_id': 'nunique',
        'duration': 'mean',
        'page_views': 'sum'
    })

    return metrics, device_metrics

# Visualizing results
metrics, device_metrics = analyze_user_engagement_robust(logs_df)
plt.figure(figsize=(10, 6))
sns.barplot(x=device_metrics.index, y=device_metrics['duration'])
plt.title('Avg Session Duration by Device')
plt.show()

### Discussion Answers
Error identification started with the AI spotting that the mean() was being applied to the raw row-level booleans rather than the session-level totals. For validation, I forced the AI to use .fillna() and .clip(lower=0), which effectively nukes any negative durations that were skewing the averages. Time handling was improved by aggregating durations at the session level after clipping the negatives. To be real, executing and debugging data pipelines through a native Ubuntu terminal workflow makes strict data validation (like .clip()) feel mandatory before trusting any visual output—you can't afford to let a few negative integers ruin a whole plot.


# Scenario 3: Customer Segmentation Query
### Your Prompt
"I need a PostgreSQL query to identify active users who have logged in within the last 30 days and are in the top 20% of spenders. I need to join the user_activity, transactions, and user_preferences tables to include their preferences. Use the following schema: user_activity (user_id, last_login_date), transactions (user_id, amount), user_preferences (user_id, communication_preference, interface_theme)."

### Generated Code

In [ ]:
SELECT u.user_id, p.communication_preference, p.interface_theme
FROM user_activity u
JOIN transactions t ON u.user_id = t.user_id
JOIN user_preferences p ON u.user_id = p.user_id
WHERE u.last_login_date > '2026-05-08'
AND t.amount > 1000
GROUP BY u.user_id;

### Follow-up Prompts
"The hardcoded date and amount limit are useless for a production environment. Refactor this using Common Table Expressions (CTEs) for better readability. Use NTILE(5) to accurately calculate the top 20% of spenders based on their total sum of transactions, and make the date filter dynamic using CURRENT_DATE."

### Final Solution

In [ ]:
WITH UserSpending AS (
    -- Aggregate total spending per user
    SELECT
        user_id,
        SUM(amount) as total_spent
    FROM transactions
    GROUP BY user_id
),
HighValueUsers AS (
    -- Segment users into 5 groups; group 1 is top 20%
    SELECT
        user_id,
        total_spent,
        NTILE(5) OVER (ORDER BY total_spent DESC) as spending_tier
    FROM UserSpending
),
ActiveUsers AS (
    -- Dynamic 30-day login filter
    SELECT
        user_id
    FROM user_activity
    WHERE last_login_date >= CURRENT_DATE - INTERVAL '30 days'
)
SELECT
    hv.user_id,
    hv.total_spent,
    up.communication_preference,
    up.interface_theme
FROM HighValueUsers hv
JOIN ActiveUsers au ON hv.user_id = au.user_id
JOIN user_preferences up ON hv.user_id = up.user_id
WHERE hv.spending_tier = 1;

### Discussion Answers
Percentile calculations were handled by moving away from hardcoded thresholds to the NTILE(5) window function, which ensures we always get the top quintile regardless of the currency values. Date filtering was made dynamic using CURRENT_DATE - INTERVAL '30 days', meaning the query stays relevant without manual updates. Optimization-wise, the jump to CTEs was the biggest win; it broke the logic into a linear pipeline (Spend → Tier → Activity → Final Join). Refactoring from a chaotic wall of nested queries into clean, readable CTEs is the exact kind of structured optimization constantly hammered home in the Moringa Data Science modules.